# Notebook 7 – Outlier Analysis
**File:** `07_Outlier_Analysis.ipynb`

## Objective
Identify unusual observations in numerical features, compare multiple detection techniques, and document a reasoned decision for each outlier. **Do not automatically remove every outlier.**

## Learning outcomes
- Explain outliers and why they occur.
- Distinguish univariate and multivariate outliers.
- Detect potential outliers with IQR and Z-score methods.
- Visualize data using box plots and scatter plots.
- Decide whether to retain, correct, remove, or transform observations based on context.

## 1. What is an outlier?
An **outlier** is an observation that is unusually far from most other observations. It may be a mistake, a rare event, or a valid extreme value.

## 2. Why do outliers occur?
- Data-entry or measurement errors
- Faulty sensors or data collection
- Different populations mixed together
- Rare but genuine events
- Natural variation in the data

## 3. Types of outliers
- **Univariate outlier:** unusual value in one feature, such as an unusually high salary.
- **Multivariate outlier:** unusual combination of values across multiple features, even when each value alone looks normal.

## 4. Load and inspect a dataset
This notebook uses a reproducible synthetic dataset so the notebook runs without downloading files. Replace it with your own dataset when ready.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the example reproducible
rng = np.random.default_rng(42)

# Synthetic customer data
n = 100
df = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "annual_spend": rng.normal(50000, 10000, n).round(0),
    "visits_per_year": rng.normal(12, 3, n).round(0).clip(1, None),
})

# Add a few deliberately unusual values for demonstration
df.loc[5, "annual_spend"] = 150000
df.loc[20, "annual_spend"] = -5000  # likely invalid for annual spend
df.loc[70, "visits_per_year"] = 45

df.head()

### Initial inspection
Check dimensions, data types, descriptive statistics, and impossible values before applying statistical rules.

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe())
print("Missing values per column:\n", df.isna().sum())

## 5. Univariate outlier detection

### IQR method
The interquartile range (IQR) measures the spread of the middle 50% of values.

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower fence = Q1 − 1.5 × IQR
- Upper fence = Q3 + 1.5 × IQR

Values outside these fences are **potential** outliers, not automatically errors.

In [ ]:
def iqr_outlier_report(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    report = data.loc[mask, ["customer_id", column]].copy()
    report["outlier_reason"] = np.where(
        report[column] < lower, "Below lower IQR fence", "Above upper IQR fence"
    )
    return {"q1": q1, "q3": q3, "iqr": iqr, "lower_fence": lower,
            "upper_fence": upper, "outliers": report}

for col in ["annual_spend", "visits_per_year"]:
    result = iqr_outlier_report(df, col)
    print(f"\n{col}: lower={result['lower_fence']:.2f}, upper={result['upper_fence']:.2f}")
    display(result["outliers"])

### Z-score method
A Z-score tells us how many standard deviations a value is from the mean.

Common screening rule: values with absolute Z-score greater than 3 may be flagged. This rule works best for roughly normal distributions and can be distorted by extreme values.

In [ ]:
def zscore_outlier_report(data, column, threshold=3):
    values = data[column]
    mean = values.mean()
    std = values.std(ddof=0)
    if std == 0:
        return data.iloc[0:0][["customer_id", column]].assign(z_score=pd.Series(dtype=float))
    z = (values - mean) / std
    mask = z.abs() > threshold
    result = data.loc[mask, ["customer_id", column]].copy()
    result["z_score"] = z.loc[mask].round(3)
    return result

for col in ["annual_spend", "visits_per_year"]:
    print(f"Potential Z-score outliers in {col}:")
    display(zscore_outlier_report(df, col))

## 6. Visual detection

### Box plot
A box plot summarizes the median, quartiles, spread, and possible outliers.

### Scatter plot
A scatter plot can reveal unusual relationships between two numerical features. A point may be a multivariate outlier if its combination of values is unusual.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df["annual_spend"].dropna(), vert=True)
axes[0].set_title("Box Plot: Annual Spend")
axes[0].set_ylabel("Annual spend")

axes[1].boxplot(df["visits_per_year"].dropna(), vert=True)
axes[1].set_title("Box Plot: Visits per Year")
axes[1].set_ylabel("Visits")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["annual_spend"], df["visits_per_year"], alpha=0.75)
plt.xlabel("Annual spend")
plt.ylabel("Visits per year")
plt.title("Scatter Plot: Annual Spend vs Visits")
plt.grid(True, alpha=0.25)
plt.show()

## 7. Multivariate outliers
A multivariate outlier is unusual because of the **combination** of multiple features. A scatter plot is a useful first check for two features. For larger datasets, methods such as Isolation Forest, Local Outlier Factor, or robust covariance can be explored, after considering scaling and domain context.

The plot below uses a simple distance-from-center score only as a teaching aid; it is not a universal outlier rule.

In [ ]:
features = ["annual_spend", "visits_per_year"]
X = df[features].astype(float)

# Standardize features so their units do not dominate the distance
stds = X.std(ddof=0).replace(0, 1)
Z = (X - X.mean()) / stds
distance = np.sqrt((Z ** 2).sum(axis=1))
df["standardized_distance"] = distance

# Illustrative flag only; review points in context before acting
distance_threshold = distance.quantile(0.99)
df["multivariate_candidate"] = distance > distance_threshold

display(df.loc[df["multivariate_candidate"], ["customer_id"] + features + ["standardized_distance"]])

## 8. Outlier review and decision log

For every flagged value, ask:
1. Is it a genuine observation?
2. Is it a data-entry or measurement error?
3. Is it unusual but valid?
4. Should it be removed, corrected, transformed, or retained?

**Example decisions (based on the synthetic data):**
- `annual_spend = -5000`: likely invalid if this feature means annual customer spending. Verify against the source; correct only if the correct value is known. Otherwise mark missing or exclude with a documented reason.
- `annual_spend = 150000`: could be a genuine high-value customer. Verify with source/domain context; retain if valid.
- `visits_per_year = 45`: unusual but may be valid. Check the definition and source before deciding.

Do not make final decisions based only on an IQR or Z-score flag.

In [ ]:
# Example decision log: update after checking the real source data.
decision_log = pd.DataFrame([
    {
        "customer_id": 21,
        "feature": "annual_spend",
        "value": -5000,
        "possible_issue": "Impossible/invalid value for annual spend",
        "verification_needed": "Check original record and feature definition",
        "decision": "Investigate; do not automatically delete",
        "reason": "Negative spend may indicate an entry error or a different accounting meaning",
    },
    {
        "customer_id": 6,
        "feature": "annual_spend",
        "value": 150000,
        "possible_issue": "High value",
        "verification_needed": "Check transaction/source record",
        "decision": "Retain if verified",
        "reason": "Could represent a genuine high-spending customer",
    },
    {
        "customer_id": 71,
        "feature": "visits_per_year",
        "value": 45,
        "possible_issue": "High visit count",
        "verification_needed": "Check definition and source",
        "decision": "Investigate, then retain if valid",
        "reason": "May be an unusually active customer",
    },
])
display(decision_log)

## 9. Optional handling examples

Only use a handling approach after validating the data and documenting the reason.

- **Correct:** only when the source confirms the correct value.
- **Remove:** when the record is confirmed invalid and cannot be repaired, and exclusion is justified.
- **Transform:** log transformation can reduce right-skew for positive values.
- **Cap/winsorize:** limit extremes only when justified by the business problem and documented.
- **Retain:** keep valid rare events, especially when they matter to the analysis.

Example log transform for strictly positive annual spend:

In [ ]:
positive_spend = df.loc[df["annual_spend"] > 0, "annual_spend"]
if not positive_spend.empty:
    log_spend = np.log1p(positive_spend)
    log_spend.hist(bins=20)
    plt.title("Log-transformed Annual Spend (positive values only)")
    plt.xlabel("log1p(annual spend)")
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No positive values available for log transformation.")

## 10. Findings and conclusion
Write your findings after running the notebook on your dataset.

- Which features had potential outliers?
- Did IQR and Z-score identify the same records?
- What did the box plot and scatter plot reveal?
- Which values were confirmed as errors?
- Which unusual values were valid and retained?
- What handling decisions were made, and why?

### Conclusion
Outlier detection identifies values that deserve investigation. Statistical flags are not proof of bad data. Domain knowledge, source verification, and a documented decision are required before changing the dataset.

In [ ]:
# Fill this in after analyzing your actual dataset.
findings = {
    "features_with_potential_outliers": [],
    "confirmed_data_errors": [],
    "valid_unusual_observations_retained": [],
    "handling_decisions_and_reasons": [],
}
findings